# SRS Lipid Droplet Analysis Using a Pretrained StarDist Model

This notebook applies the pretrained `lipid_droplet_v1` StarDist model
to new stimulated Raman scattering (SRS) microscopy images.

**No model training is required.**

The workflow is:

New SRS images  
→ Intensity normalization  
→ Pretrained StarDist model  
→ Lipid-droplet instance segmentation  
→ Quantitative analysis  
→ Export masks, overlays, and CSV files

## Outputs

For each SRS image, this notebook generates:

- Instance-segmentation mask (`.tif`)
- Prediction overlay (`.png`)
- Individual lipid-droplet measurements (`.csv`)

A combined summary CSV is also generated for all analyzed images.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tifffile import imread, imwrite
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from skimage.measure import regionprops_table

print("Packages loaded successfully.")

## User Settings

Edit the settings below before running the analysis.

### Pixel size

`PIXEL_SIZE_UM` should be the physical size represented by one pixel
in the SRS image, in µm/pixel.

For example:

```python
PIXEL_SIZE_UM = 0.293

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

MODEL_NAME = "lipid_droplet_v1"

# Physical pixel size in µm/pixel.
# IMPORTANT: Change this to match your imaging conditions.
PIXEL_SIZE_UM = 0.293

# Supported input image extensions
IMAGE_EXTENSIONS = (
    "*.tif",
    "*.tiff"
)

print("Model:", MODEL_NAME)
print("Pixel size:", PIXEL_SIZE_UM, "µm/pixel")

In [ ]:
# ============================================================
# PROJECT DIRECTORIES
# ============================================================

# This notebook should be located inside:
# SRS_LipidDroplet_StarDist/notebooks/

PROJECT_DIR = Path("..")

MODEL_DIR = PROJECT_DIR / "model"
IMAGE_DIR = PROJECT_DIR / "images"
RESULT_DIR = PROJECT_DIR / "results"

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory :", MODEL_DIR.resolve())
print("Image directory :", IMAGE_DIR.resolve())
print("Results directory:", RESULT_DIR.resolve())

## Load the Pretrained Model

The following cell loads the pretrained `lipid_droplet_v1` model.

It does not perform any training.

In [ ]:
# ============================================================
# LOAD PRETRAINED MODEL
# ============================================================

model_path = MODEL_DIR / MODEL_NAME

if not model_path.exists():

    raise FileNotFoundError(
        f"Model folder not found:\n"
        f"{model_path.resolve()}\n\n"
        f"Place the complete '{MODEL_NAME}' folder "
        f"inside the repository's model/ directory."
    )


model = StarDist2D(
    None,
    name=MODEL_NAME,
    basedir=str(MODEL_DIR)
)

print(
    f"Model '{MODEL_NAME}' loaded successfully."
)

## Find Input Images

Place the SRS TIFF images that you want to analyze inside the
`images/` directory.

All `.tif` and `.tiff` images in that directory will be processed.

In [ ]:
# ============================================================
# FIND INPUT IMAGES
# ============================================================

image_files = []

for extension in IMAGE_EXTENSIONS:

    image_files.extend(
        IMAGE_DIR.glob(extension)
    )


image_files = sorted(
    set(image_files)
)


print(
    f"Found {len(image_files)} image(s).\n"
)


for i, image_path in enumerate(
    image_files,
    start=1
):

    print(
        f"{i}: {image_path.name}"
    )


if len(image_files) == 0:

    raise FileNotFoundError(
        f"No TIFF images were found in:\n"
        f"{IMAGE_DIR.resolve()}"
    )

In [ ]:
# ============================================================
# TEST MODEL ON FIRST IMAGE
# ============================================================

test_path = image_files[0]

test_img = imread(
    test_path
)


if test_img.ndim != 2:

    raise ValueError(
        f"{test_path.name} has shape {test_img.shape}. "
        "This workflow expects a single-channel 2D SRS image."
    )


# Normalize using the same method used during training

test_img_norm = normalize(
    test_img,
    1,
    99.8
)


# Predict lipid droplets

test_mask, details = model.predict_instances(
    test_img_norm
)


droplet_count = len(
    np.unique(test_mask)[
        np.unique(test_mask) > 0
    ]
)


print(
    "Test image:",
    test_path.name
)

print(
    "Detected lipid droplets:",
    droplet_count
)

In [ ]:
# ============================================================
# VISUALIZE TEST PREDICTION
# ============================================================

overlay = np.ma.masked_where(
    test_mask == 0,
    test_mask
)


plt.figure(
    figsize=(14, 6)
)


# Original SRS image

plt.subplot(
    1,
    2,
    1
)

plt.imshow(
    test_img,
    cmap="gray"
)

plt.title(
    "Original SRS Image"
)

plt.axis("off")


# Prediction

plt.subplot(
    1,
    2,
    2
)

plt.imshow(
    test_img,
    cmap="gray"
)

plt.imshow(
    overlay,
    cmap="nipy_spectral",
    alpha=0.5
)

plt.title(
    f"StarDist Prediction\n"
    f"{droplet_count} lipid droplets"
)

plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MEASURE INDIVIDUAL LIPID DROPLETS
# ============================================================

measurements = regionprops_table(
    test_mask,
    properties=[
        "label",
        "area",
        "equivalent_diameter_area",
        "centroid"
    ]
)


test_df = pd.DataFrame(
    measurements
)


# Rename columns for clarity

test_df = test_df.rename(
    columns={
        "area": "area_px2",
        "equivalent_diameter_area":
            "equivalent_diameter_px",
        "centroid-0": "centroid_y_px",
        "centroid-1": "centroid_x_px"
    }
)


# Convert to physical units if calibration is available

if PIXEL_SIZE_UM is not None:

    test_df["area_um2"] = (
        test_df["area_px2"]
        * PIXEL_SIZE_UM**2
    )

    test_df["equivalent_diameter_um"] = (
        test_df["equivalent_diameter_px"]
        * PIXEL_SIZE_UM
    )


display(
    test_df.head()
)


print(
    "\nDroplet number:",
    len(test_df)
)


if PIXEL_SIZE_UM is not None:

    print(
        "Total lipid-droplet area:",
        test_df["area_um2"].sum(),
        "µm²"
    )

    print(
        "Mean lipid-droplet area:",
        test_df["area_um2"].mean(),
        "µm²"
    )

In [ ]:
# ============================================================
# BATCH-PROCESSING FUNCTION
# ============================================================

def analyze_image(image_path):

    """
    Analyze one SRS image using the pretrained StarDist model.

    Outputs:
        - Instance segmentation TIFF
        - Prediction overlay PNG
        - Per-droplet CSV

    Returns:
        Dictionary containing image-level summary measurements.
    """

    print(
        f"Processing: {image_path.name}"
    )


    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    img = imread(
        image_path
    )


    if img.ndim != 2:

        raise ValueError(
            f"{image_path.name} has shape {img.shape}. "
            "Expected a single-channel 2D image."
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    img_norm = normalize(
        img,
        1,
        99.8
    )


    # --------------------------------------------------------
    # StarDist prediction
    # --------------------------------------------------------

    labels, details = model.predict_instances(
        img_norm
    )


    # --------------------------------------------------------
    # Save instance mask
    # --------------------------------------------------------

    mask_path = (
        RESULT_DIR
        / f"{image_path.stem}_mask.tif"
    )


    if labels.max() <= np.iinfo(np.uint16).max:

        labels_to_save = labels.astype(
            np.uint16
        )

    else:

        labels_to_save = labels.astype(
            np.uint32
        )


    imwrite(
        mask_path,
        labels_to_save
    )


    # --------------------------------------------------------
    # Measure droplets
    # --------------------------------------------------------

    measurements = regionprops_table(
        labels,
        properties=[
            "label",
            "area",
            "equivalent_diameter_area",
            "centroid"
        ]
    )


    df = pd.DataFrame(
        measurements
    )


    df = df.rename(
        columns={
            "area": "area_px2",
            "equivalent_diameter_area":
                "equivalent_diameter_px",
            "centroid-0": "centroid_y_px",
            "centroid-1": "centroid_x_px"
        }
    )


    df.insert(
        0,
        "image",
        image_path.name
    )


    # --------------------------------------------------------
    # Convert to physical units
    # --------------------------------------------------------

    if PIXEL_SIZE_UM is not None:

        df["area_um2"] = (
            df["area_px2"]
            * PIXEL_SIZE_UM**2
        )

        df["equivalent_diameter_um"] = (
            df["equivalent_diameter_px"]
            * PIXEL_SIZE_UM
        )


    # --------------------------------------------------------
    # Save individual-droplet measurements
    # --------------------------------------------------------

    droplet_csv = (
        RESULT_DIR
        / f"{image_path.stem}_droplets.csv"
    )


    df.to_csv(
        droplet_csv,
        index=False
    )


    # --------------------------------------------------------
    # Save prediction overlay
    # --------------------------------------------------------

    overlay = np.ma.masked_where(
        labels == 0,
        labels
    )


    fig = plt.figure(
        figsize=(8, 8)
    )


    plt.imshow(
        img,
        cmap="gray"
    )


    plt.imshow(
        overlay,
        cmap="nipy_spectral",
        alpha=0.5
    )


    plt.title(
        f"{image_path.name}\n"
        f"Detected droplets: {len(df)}"
    )


    plt.axis("off")

    plt.tight_layout()


    overlay_path = (
        RESULT_DIR
        / f"{image_path.stem}_overlay.png"
    )


    fig.savefig(
        overlay_path,
        dpi=200,
        bbox_inches="tight"
    )


    plt.close(fig)


    # --------------------------------------------------------
    # Image-level summary
    # --------------------------------------------------------

    summary = {

        "image":
            image_path.name,

        "droplet_count":
            len(df),

        "total_droplet_area_px2":
            df["area_px2"].sum()
            if len(df) else 0,

        "mean_droplet_area_px2":
            df["area_px2"].mean()
            if len(df) else np.nan,

        "median_droplet_area_px2":
            df["area_px2"].median()
            if len(df) else np.nan,

        "mean_droplet_diameter_px":
            df["equivalent_diameter_px"].mean()
            if len(df) else np.nan
    }


    if PIXEL_SIZE_UM is not None:

        summary.update({

            "total_droplet_area_um2":
                df["area_um2"].sum()
                if len(df) else 0,

            "mean_droplet_area_um2":
                df["area_um2"].mean()
                if len(df) else np.nan,

            "median_droplet_area_um2":
                df["area_um2"].median()
                if len(df) else np.nan,

            "mean_droplet_diameter_um":
                df["equivalent_diameter_um"].mean()
                if len(df) else np.nan
        })


    return summary

In [ ]:
# ============================================================
# ANALYZE ALL IMAGES
# ============================================================

summary_rows = []


for image_path in image_files:

    summary = analyze_image(
        image_path
    )

    summary_rows.append(
        summary
    )


print(
    "\nAll images processed successfully."
)

In [ ]:
# ============================================================
# CREATE EXPERIMENT SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


summary_path = (
    RESULT_DIR
    / "lipid_droplet_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


display(
    summary_df
)


print(
    "\nSummary saved to:"
)

print(
    summary_path.resolve()
)

# Output Files

For every analyzed SRS image, the `results/` directory contains:

### `*_mask.tif`

StarDist instance-segmentation mask.

- 0 = background
- 1 = lipid droplet 1
- 2 = lipid droplet 2
- etc.

### `*_overlay.png`

Quality-control image showing predicted lipid-droplet masks over the
original SRS image.

### `*_droplets.csv`

One row represents one detected lipid droplet.

Measurements include:

- droplet label
- area in pixels²
- equivalent diameter in pixels
- centroid position
- area in µm² (when pixel calibration is supplied)
- equivalent diameter in µm

### `lipid_droplet_summary.csv`

One row represents one SRS image.

The summary includes:

- lipid-droplet count
- total lipid-droplet area
- mean lipid-droplet area
- median lipid-droplet area
- mean equivalent droplet diameter

# Quality Control

Prediction overlays should be visually inspected before quantitative
results are used for biological interpretation.

Particular attention should be paid to:

- missed lipid droplets,
- false-positive detections,
- merged neighboring droplets,
- split droplets,
- inaccurate segmentation boundaries,
- unusually large or small lipid droplets.

The current model was trained using a limited SRS dataset. Performance
on images acquired using substantially different imaging conditions
should be independently validated.